In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

# Creating Schema
spark.sql("CREATE SCHEMA IF NOT EXISTS novocart_global.novocart_bronze")

# Tables with their primary keys
table_config = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "exchange_rates": "currency_code"
}

for tbl, pk in table_config.items():
    
    print(f"Table in process: {tbl}")
    
    # Reading from source (table)
    df = spark.table(f"novocart_global.azure_blob_storage.{tbl}")
    
    # Adding Audit column (UC supported)
    df = df.withColumn("ingestion_time", current_timestamp())
    
    target_table = f"novocart_global.novocart_bronze.{tbl}_raw"
    
    # Checking if table exists
    if spark.catalog.tableExists(target_table):
        
        delta_table = DeltaTable.forName(spark, target_table)
        
        merge_condition = f"tgt.{pk} = src.{pk}"
        
        # Incremental load 
        delta_table.alias("tgt").merge(
            df.alias("src"),
            merge_condition
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"Incremental load completed for {tbl}")
    
    else:
        # First time load
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)
        
        print(f"Initial load completed for {tbl}")